> **💾 First: save your own copy** · This notebook opens from the course website, so Colab does not keep your changes. Before you do anything else, go to **File → Save a copy in Drive**. Colab opens your copy in a new tab: work in that tab and close this one. You find your copy later in Google Drive, in the folder *Colab Notebooks*.

> **⚠️ Important** · Before you run any cells in this notebook, in the top menu go to **Runtime|Change runtime type** and switch your runtime to **Python 3** with hardware accelerator **T4 GPU**.
>
> When asked, please **allow the notebook access to Google Drive**: choose the same Google account you use for Colab and tick all the permissions. Drive is used to read the models from the shared course cache (the CDA-CACHE shortcut you set up following the instructions on Canvas).
>
> Please run the cell below to set up the environment. This takes a little while.
>
> **This workshop consists of three parts. The first part, 'Case Study: Sentiment analysis of social media', covers hate-speech detection and fine-tuning Transformers. The second part, 'A Second Case Study: Do Two Sentences Mean the Same Thing', is about text-to-text Transformers, which are the reason why people now also use LLMs for tasks like sentiment analysis. The third part, 'A Tour of Transformer Applications', runs through other so-called zero-shot applications with Transformers. To get a complete, please complete 2 of the 3 parts.**


In [ ]:
# Keep cell
# Running this takes a while!
%%capture

import sys
import os
import nltk

import matplotlib.pyplot as plt
import torch

import pandas as pd
import numpy as np

try:
  import learntools.core
  from learntools.core import binder
  binder.bind(globals())
except:
  !pip install git+https://github.com/Kaggle/learntools.git
  import learntools.core
  from learntools.core import binder
  binder.bind(globals())

if not(os.path.exists('text_workshop_tests.py')):
  print('Downloading text_workshop_tests.py\n')
  !wget https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/text_workshop_tests.py


from text_workshop_tests import *


#Load models ...

# The models come from the shared course cache (the CDA-CACHE shortcut in your Google Drive),
# so that not everyone has to download them from Hugging Face at the same time.
# The shared cache is read-only for you. Everything Hugging Face needs to write
# (datasets, temporary files) therefore stays on this Colab machine.
LOCAL_HF = "/content/hf-local"
SHARED_HUB = "/content/drive/MyDrive/CDA-CACHE/huggingface/hub"
os.makedirs(LOCAL_HF, exist_ok=True)
os.environ['HF_HOME'] = LOCAL_HF

try:
  from google.colab import drive
  drive.mount('/content/drive')
except Exception as e:
  print("Google Drive not mounted.", e)

if os.path.isdir(SHARED_HUB):
  os.environ['HF_HUB_CACHE'] = SHARED_HUB
  print("Using the shared course cache for the models.")
else:
  print("The CDA-CACHE shortcut was not found in your Google Drive - see the setup instructions on Canvas.")
  print("The models will be downloaded for this session only.")

import transformers

try:
  import datasets
except:
  !pip install datasets
  import datasets

try:
  import accelerate
except:
  !pip install accelerate
  import accelerate

try:
  nltk.data.find('tokenizers/punkt_tab')
except LookupError:
  nltk.download('punkt')
  nltk.download('punkt_tab')

try:
  nltk.data.find('corpora/stopwords')
except LookupError:
  nltk.download('stopwords')


if not(os.path.exists('TwitterHate_Small.csv')):
  print('Downloading TwitterHate_Small.csv')
  !wget https://raw.githubusercontent.com/goto4711/cdai/main/TwitterHate_Small.csv



is_colab = "google.colab" in sys.modules
is_gpu_available = torch.cuda.is_available()

def display_library_version(library):
    print(f"Using {library.__name__} v{library.__version__}")

# Give visibility on versions of the core libraries
display_library_version(transformers)
display_library_version(datasets)
# Disable all info / warning messages
transformers.logging.set_verbosity_error()
datasets.logging.set_verbosity_error()


from transformers import set_seed
set_seed(42) # Set the seed to get reproducible results

if not is_gpu_available:
    print('------' * 10)
    print("No GPU was detected! This notebook can be *very* slow without a GPU 🐢")
    if is_colab:
        print("Go to Runtime > Change runtime type and select a GPU hardware accelerator.")


![Cultural Data Analysis · Week 4: Cultural Data Forms: Text](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week4-banner.png)

# Cultural Data Forms - Texts

*Workshop notebook* · Code-along: [goto4711.github.io/cda-teaching-tools/code-along/week4](https://goto4711.github.io/cda-teaching-tools/code-along/week4/) · Tools: [term frequency](https://goto4711.github.io/cda-teaching-tools/tools/text/), [from words to embeddings](https://goto4711.github.io/cda-teaching-tools/tools/embeddings/)

---

In this workshop, we will continue examining hate-speech detection. As we have seen, our simple methods have failed to deliver convincing results using dictionaries. Transformers (https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) are the foundation of some of the most popular AI applications like ChatGPT. Will they do better? Don't worry, it is not as complicated as it sounds.

With Transformers, machines can generate text, translate from one language to another, analyse comments, label words in sentences, etc. Transformers have been introduced in a paper called 'Attention is all you need' (https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf). It has become one of the coining papers of the new AI age.

There is a lot of critical work on monopolisation in the field of AI, with Microsoft a major investor in OpenAI and also the owner of the cloud infrastructure that runs ChatGPT applications, etc. One attempt to work against this is to create open-source versions of deep-learning architectures and models. A leading organisation in this field of open-source AI is Hugging Face, which provides a wide range of models for text analysis or so-called language models (https://huggingface.co/). They make interacting with Transformers quite easy, as you will find out by the end of this workshop. In many ways, the Hugging Face way of doing NLP is easier than what you have seen in the lecture.

> **Goals** · By the end of this workshop, you will understand the Hugging Face ecosystem, before you learn how to fine-tune a Transformers model. Then at the end, we find out how easy it all is by using Hugging Face directly without any further training.  The last part 'Tour of Transformer Applications' (zero-shot) shows the breadth of what is possible but you can be selective in what you actually run. If time is tight, you can at first ignore the second case study on paraphrase detection.

We can do all the things we introduced in the lecture easier (and often much better) with Transformers from classification to information extraction. Before we reach the easy things, we need to, however, first understand more about the Hugging Face ecosystem. It allows you to load datasets of texts and transform them so that they can be used in deep learning. To this end, it introduces much more efficient tokenizers than we have already seen. Most importantly, Hugging Face's models help us reuse the knowledge stored in other language models. This process is called fine-tuning and means that we can take an existing language model and adjust it to our work. For instance, we could pick up the 'BERT' language model based on the original Transformers paper and fine-tune it so that it would be able to recognise sentiments in the state of the unions.

The figure below shows the Hugging Face pipeline, which we will explore in this workshop. As always, once we have learned the pipeline steps we can build many different applications with it.

<img alt="Hugging Face Pipeline" caption="A typical pipeline for training transformer models with the Datasets, Tokenizers and Transformers" src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg"/>

We will employ Transformers to predict whether an input text is hate speech or not based on examples we will train a model with. Thus, while in the lecture we used unsupervised learning with dictionaries to detect hate speech, we now employ supervised learning. This requires training datasets that are labelled, which we can also get from Kaggle and Hugging Face. To make it more interesting, we switch to more relevant data for this kind of work and look at hate speech in Tweets and social media.

## Case Study: Sentiment analysis of social media

For the sentiment analysis of Tweets, we will use a dataset from the Kaggle platform, which we have already met a couple of times. If you use Kaggle for nothing else in this course, you might find it interesting to look into their datasets. Check out https://www.kaggle.com/datasets to try out your newly gained data skills.

Today, we use a simple dataset for Twitter hate speech https://www.kaggle.com/datasets/prakharprasad/twitter-hate-speech that contains several thousand Tweets and classifies them as either hate speech or not. Please note that we reduce the size of the data to make it easier to analyse in this workshop.

Before we move to our Twitter dataset, let's see how Hugging Face deals with data in general.

### A First Look at Hugging Face Datasets

The Hugging Face data hub contains many datasets (https://huggingface.co/datasets). We can use the `list_datasets()` function from the main Hugging Face Hub library to see which datasets are available.

Run the cell below and ignore the warnings.

In [ ]:
#Keep cell
#Takes a while - please be patient
#Commented out as you might want to skip this cell and the next one. They take a long time and are more for information only.
#Uncomment the code, if you are interested.

from huggingface_hub import list_datasets
# all_datasets = list(list_datasets())

# print(f"There are {len(all_datasets)} datasets currently available on the Hub.")
# print(f"The first 5 are: {all_datasets[:5]}")

We can see that each dataset is given a name/id, so let's see which one looks promising for hate-speech detection. Run the next cell.

In [ ]:
#Keep cell
#Commented out as you might want to skip this cell and the previous one. They take a long time and are more for information only.
#Uncomment the code, if you are interested.

# for s in all_datasets:
#   s_ = os.path.basename(s.id)
#   if "hate" in s_:
#     print(s_)

That's a lot of datasets. There is a lot of research on hate speech. Let's load one of them: `tdavidson/hate_speech_offensive`.

Run the cell below.

In [ ]:
#Keep cell
from datasets import load_dataset
hatespeech_hf = load_dataset("tdavidson/hate_speech_offensive")

Inside `hatespeech_hf`, we find a so-called `DatasetDict`. Print out `hatespeech_hf`.

This Hugging Face creation looks similar to a Python dictionary, with the key of `train` corresponding to a training dataset of over 24k rows at the time of writing. Let's index into `train`. Run
```
train_ds = hatespeech_hf["train"]
train_ds
```

Let's run a test that everything is set up ok.

In [ ]:
# Run this cell to test your code...
assert len(train_ds) > 24760, f'Test failed'
print('All tests passed!')

Now, we have our `Dataset` object as the core structure in datasets. Among other things, it behaves like a Python list. Try and run `len` with `train_ds`.

We can also access a single example by its index. Type in `train_ds[0]`.

The documentation for this dataset can be found at https://huggingface.co/datasets/tdavidson/hate_speech_offensive/. Thus the above keys indicate:

- count: (Integer) number of users who labelled each Tweet - min is 3, sometimes more users coded a Tweet when judgements were determined to be unreliable.
- hate_speech_count: (Integer) number of users who judged the Tweet to be hate speech.
- offensive_language_count: (Integer) number of users who judged the Tweet to be offensive.
- neither_count: (Integer) number of users who judged the Tweet to be neither hate speech nor offensive.
- class: (Class Label) class label for majority of CF users (0: 'hate-speech', 1: 'offensive-language' or 2: 'neither').
- tweet: (string).

Run `train_ds.column_names`.

The data types for each column can be found in the `features` attribute of `train_ds`. Do you know how to print it out? A quick online search will also help.

In [ ]:
q0.hint()
#q0.solution()

So, `count`, for instance, is an integer, while `tweet` is a string. Let's print out the first three elements with `train_ds[:3]`.

The dictionary values are lists instead of individual elements.

We can also go to the full key by name. The first five Tweets can be printed with `train_ds["tweet"][:5]`. Try it.

### What if my dataset is not on the hub?

Most often, a dataset will not be on the hub. The hate speech dataset is actually a version of a famous dataset on Kaggle, as far as I can see https://www.kaggle.com/code/prakharprasad/twitter-hate-speech-classification. Somebody has imported it to Hugging Face.

Let's look at the simpler hate-speech dataset at https://www.kaggle.com/datasets/prakharprasad/twitter-hate-speech to understand how to import it into our processing with Hugging Face. We have downloaded a smaller version of it for you to make it a bit more balanced and easier to process with Hugging Face's dataset tools: https://huggingface.co/docs/datasets/en/process. Label is either 1 for a Tweet containing hate speech or 0 otherwise. 'tweet' contains the Tweet.

We can load the local dataset with `load_dataset` by using the `csv` script and pointing the `data_files` argument to `"TwitterHate_Small.csv"`. The column names (id, label, tweet) are read from the first line of the file. `class_encode_column('label')` tells the Transformers to use 'label' as a target column.

Run the next cell.

In [ ]:
#Keep cell
ds = load_dataset("csv", data_files="TwitterHate_Small.csv")
ds = ds.class_encode_column('label')

The `load_dataset()` function is very powerful: https://huggingface.co/docs/datasets/en/loading. Let's take a look at the data next and print out `ds` and some data (the first 5 entries). Complete the cell below.

In [ ]:
#Complete cell
print(ds)
print(___['train'][:5])

In [ ]:
q1.hint()
#q1.solution()

As we are experts in machine learning by now, we know that we need a test dataset next to the training dataset. Actually, we would have also needed a validation dataset to make sure that our algorithm is good. It allows us to use the test data only for a final test and tune the model with the validation set. But we skip this here for learning reasons. You can read up on this at https://en.wikipedia.org/wiki/Training,_validation,_and_test_data_sets.

For the training and test data, we choose an 80-20 random split by running the cell below. Do you see how? Do you remember how we did this for sklearn? For the Hugging Face version with its `train_test_split()`, run the next cell.

In [ ]:
#Keep cell
hatespeech_local = ds['train']
hatespeech_local = hatespeech_local.train_test_split(test_size=0.2, seed = 42)
hatespeech_local

### From Datasets to DataFrames

Pandas is great for analysing data and visualising it. We can convert a `Dataset` to a Pandas `DataFrame` with the `set_format()` method. Run the cell below.

In [ ]:
#Keep cell
import pandas as pd
import numpy as np

hatespeech_local.set_format(type="pandas")
df_hs = hatespeech_local["train"][:]
df_hs.head()

Let's run a quick check whether everything is loaded.

In [ ]:
# Run this cell to test your code...
assert df_hs.shape[0] == 6548, f'Test failed'
print('All tests passed!')

As a reminder, label 0 stands for non-hateful and 1 for hateful. Let's add a column 'label_name' to reflect this. Do you remember `np.where`? Complete the cell below.

In [ ]:
#Complete cell
df_hs['label_name'] = np.where(df_hs['label'] == ___, 'Non-hateful', 'Hateful')
df_hs.head()

In [ ]:
q2.check()
#q2.hint()
#q2.solution()

> **💡 Remember** · Whenever you are working with NLP classifications or classifications in general, it is good to start with understanding the distribution across the target classes. We cannot cover this here, but skewed classes have to be addressed in very different ways from more even distributions.

Let's visualise `label_name`, which is our target variable. Complete the cell below.

In [ ]:
#Complete cell

import matplotlib.pyplot as ___

df_hs["label_name"].value_counts(ascending=True).plot.barh()
plt.title("Frequency of Classes")
plt.___()

In [ ]:
q3.hint()
#q3.solution()

In this case, we can see that the dataset is heavily imbalanced. We could try and find more hateful Tweet examples, but we skip this here. Instead, let's look at how long the Tweets are. This is important for several reasons. We have already discussed the impact of text lengths in the lecture but for language models there is also an important additional reason. Transformer models have a maximum input sequence length that is referred to as the _maximum context size_. This is what they can see from a text at any particular moment in time, because of the way that they have been trained.

Let's look at a leaner version of the already mentioned BERT language model called DistilBERT (https://huggingface.co/docs/transformers/en/model_doc/distilbert). According to this documentation, its maximum context size is 512 tokens, which amounts to a few paragraphs of text. This should be fine for Tweets but let's check. Complete the cell below to visualise the Tweet lengths.

In [ ]:
#Complete cell

df_hs["Words Per Tweet"] = df_hs[___].str.split().apply(len)
df_hs.boxplot("Words Per Tweet", by="label_name", grid=False, color="black")
plt.suptitle("")
plt.xlabel("")
plt.show()

In [ ]:
q4.hint()
#q4.solution()

Most Tweets are around 15 words long and even the longest Tweets are well below DistilBERT's maximum context size. If that were not the case, we would have to apply specific methods to reduce the lengths of the input texts. Or we 'chunk' the texts into smaller parts: https://cobusgreyling.medium.com/chunking-for-llms-using-haystack-huggingface-81255776d71.

Before we do anything else, we have to reset the output format of our dataset since we don't need the dataframe-format anymore. Run `hatespeech_local.reset_format()`.

We have already talked about tokens/tokenization a bit in the lecture but what are they in the Transformers world? Let's find out next.

---

## From Text to Tokens

As said, tokenization is the second important step for Transformers after loading the datasets. Models like DistilBERT cannot process strings but need  _tokenized_ texts that are _encoded_ as numerical vectors. Tokenization is the step of breaking down a string into the atomic units used in the model. There are several tokenization strategies one can adopt, and the optimal splitting of words into sub-units is learned from the corpus nowadays.

Let's consider the two cases, which have been most common before Transformers: _character_ and _word_ tokenization.

### Character Tokenization

The simplest tokenization scheme is to feed each character individually to the model. In Python, `str` objects are implemented as collections of characters, which allows us to quickly implement character-level tokenization with just one line of code. Run the cell below.

In [ ]:
#Keep cell
text = "Tokenizing text is fun but also strange. Even poetry becomes a number!"
tokenized_text = list(text)
print(tokenized_text)

Afterwards, we need to convert each character into an integer, a process  called _numericalization_. Deep learning/transformers requires all input to be numbers. Numericalisation is easy for characters by simply assigning them a unique integer. The code below is typical for Transformers. It creates a dictionary, which maps tokens (here characters) to numbers by assigning an integer to them - in the order of appearance of the tokens. Run the cell.

In [ ]:
#Keep cell
token2idx = {ch: idx for idx, ch in enumerate(sorted(set(tokenized_text)))}
print(token2idx)

We can now use `token2idx` to transform the tokenized text to a list of integers. Complete the cell below.

In [ ]:
#Complete cell
input_ids = [token2idx[token] for token in ___]
print(input_ids)

In [ ]:
q5.check()
#q5.hint()
#q5.solution()

With a bit more processing, we could use these numbers based on characters for the deep-learning model. But we move on and look at word tokenization.

### Word Tokenization

Instead of splitting the text into characters, we can split it into words, as we have seen in the lecture, and map each word to an integer. One simple class of word tokenizers, we have already seen, uses whitespaces to tokenize the text. We can do this by applying Python's `split()` function directly on the raw text - just like we did to measure the Tweet lengths.

Run the cell below.

In [ ]:
#Keep cell
tokenized_text = text.split()
print(tokenized_text)

There are several problems with this approach. Most importantly maybe, it does not consider the position of a word and its context. Furthermore, the resulting vocabulary can be very large very quickly. This requires very big neural nets and makes their training very expensive.

A common mitigation approach is to limit the vocabulary and discard rarer words and only keep, say, the 100,000 most common words. Words that are not part of the limited vocabulary are classified as 'unknown' and mapped to a shared `UNK` token. You will encounter this approach quite frequently, but this also means that we lose some potentially important information in the process of word tokenization.

Wouldn't it be nice if there was a compromise between character and word tokenization that preserves all the input information _and_ some of the input structure? _Subword tokenization_ addresses this.

### Subword Tokenization

The basic idea behind subword tokenization is to combine the best aspects of character and word tokenization. On the one hand, we want to split rare words into smaller units to allow the model to deal with complex words and misspellings. On the other hand, we want to keep frequent words as unique entities so that we can keep the length of our inputs to a manageable size.

The main distinguishing feature of subword tokenization (as well as many forms of modern word tokenization) is that it is _learned_ from the pretraining corpus using a mix of statistical rules and algorithms, similar to what we have seen for Word2Vec in the lecture. Subword tokenization is a very active research field in computer science. Here, we only focus on its application, which is very easy with Hugging Face.

The subword tokenizer WordPiece (https://ieeexplore.ieee.org/document/6289079) is used by BERT and DistilBERT. The easiest way to understand how WordPiece works is to observe it in action. `transformers` provides a convenient `AutoTokenizer` class that allows you to quickly auto-load the tokenizer associated with a model. We just call the `from_pretrained()` method. We also define a so-called 'checkpoint', which one can think of as the latest save of the model. In this case, it is the latest version of the DistilBERT model https://huggingface.co/distilbert/distilbert-base-cased. This version of DistilBERT is cased: it keeps the difference between lower and upper case in English.

Run the cell below.

In [ ]:
#Keep cell
from transformers import AutoTokenizer

checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

The `AutoTokenizer` class belongs to a larger set of so-called 'auto' classes (https://huggingface.co/docs/transformers/model_doc/auto), whose job it is to automatically retrieve a model's configuration,  weights, etc. from its latest checkpoint. This allows you to quickly switch between models, but you can also load the specific model manually.

At the beginning of your learning journey, it is best to rely as much on 'auto' classes as possible. Later on, you will see that we could have also loaded the DistilBERT tokenizer directly. For instance, you could type in:
```
from transformers import DistilBertTokenizer

distilbert_tokenizer = DistilBertTokenizer.from_pretrained(checkpoint)
```
Check it out online if you want but we move on for now.

We want to see `tokenizer` in action with our earlier defined `text`. Complete the cell below.

In [ ]:
#Complete cell
encoded_text = ___(text)
print(encoded_text)

In [ ]:
q6.hint()
#q6.solution()

Every subword has now a unique integer in the `input_ids` field. You can ignore `attention_mask`. With `input_ids`, we can convert them back into tokens by using the tokenizer's `convert_ids_to_tokens()` method. Run the next cell.

In [ ]:
#Keep cell
tokens = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)
print(tokens)

Let's run a quick test that all is good.

In [ ]:
# Run this cell to test your code...
assert tokens[-1]=='[SEP]', f'Test failed'
print('All tests passed!')

From the output, we can observe three things:
1. Some special tokens, `[CLS]` and `[SEP]`, have been added to the start and end of the sequence. These tokens differ from model to model, but their main role is to indicate the start and end of a sequence.
1. The tokens have kept their upper and lower case, which is a feature of this cased version of DistilBERT.
1. Finally, we can see that the word 'Tokenizing', e.g., has been split into three tokens (`To`, `##ken`, `##izing`), since it is not a common word. The `##` prefix in `##izing` means that the preceding string is not a whitespace. With the prefix, we can convert them back.

The `AutoTokenizer` class has a `convert_tokens_to_string()` method for converting tokens back. Complete the next cell.

In [ ]:
#Complete cell
tokenizer.convert_tokens_to_string(___)

In [ ]:
q7.hint()
#q7.solution()

The `AutoTokenizer` class also has several attributes that provide information about the tokenizer. For example, we can inspect the vocabulary size or number of tokens with `tokenizer.vocab_size`. `vocab_size` is an attribute of `tokenizer` and `.` gives you access to these. Try to print out `tokenizer.vocab_size`.

The model's maximum context size or its maximum number of tokens is provided by `tokenizer.model_max_length`.

With this basic understanding of the tokenization process, let's see how we can tokenize the whole dataset.

### Tokenizing the Whole Dataset

To tokenize the whole dataset, we will use the `map()` method of our `DatasetDict` object. To this end, we need a processing function to tokenize our examples. Run the cell below to define this function.

In [ ]:
#Keep cell
def tokenize(batch):
    return tokenizer(batch["tweet"], padding=True, truncation=True)

This function applies the tokenizer to a batch of examples; `padding=True` will pad the examples with zeros to the size of the longest one in a batch, and `truncation=True` will truncate the examples to the model's maximum context size. Just ignore these for the time being. Always, set them to True if you create your own tokenizer at the beginning.

Creating your own tokenizer is simple using this example code. You just have to define the `tokenize` function in the example above for your own Tokenizer.

To see `tokenize()` in action, let's pass a batch of two examples from the training set. Run the cell below.

In [ ]:
#Keep cell
tokenize(hatespeech_local["train"][:2])

Here we can see the result of padding. The second example is shorter than the first. So, zeros have been added to that element to 'pad' it to the same length. Again, ignore `attention_mask`.

The zeros have a corresponding `[PAD]` token in the vocabulary as a special token. The set of special tokens also includes the `[CLS]` and `[SEP]` tokens that we encountered earlier. Run the cell below to see the IDs of special tokens.

In [ ]:
#Keep cell
tokens2ids = list(zip(tokenizer.all_special_tokens, tokenizer.all_special_ids))
data = sorted(tokens2ids, key=lambda x : x[-1])
pd.DataFrame(data, columns=["Special Token", "Special Token ID"]).T

With the tokenization function `tokenize(batch)`, we can apply it across all the splits in the dataset very easily with `map()`. We print the column_names of 'test' to demonstrate our success. Run:
```
hatespeech_encoded = hatespeech_local.map(tokenize, batched=True, batch_size=None)
print(hatespeech_encoded["test"].column_names)
```

Hugging Face's Transformers assume that your target variable has the column name 'labels' and the textual input is 'text'. Therefore, we need to rename the columns with `rename_columns()`. Run the cell below.

In [ ]:
#Keep cell
hatespeech_encoded = hatespeech_encoded.rename_columns({'label':'labels', 'tweet': 'text'})
print(hatespeech_encoded["test"].column_names)

This is just an exercise. So, let us reduce the training and test datasets, which is surprisingly hard with Hugging Face at the time of writing. The cell below takes care of that, by picking up 50% of the datasets. It is a bit complicated but a good exercise to try and understand the various parts. Also, it might be useful for the future when you are working with your own datasets.

Run the cell.

In [ ]:
#Keep cell
frac_ = 0.5

train_frac = int(hatespeech_encoded["train"].num_rows * frac_)
test_frac = int(hatespeech_encoded["test"].num_rows * frac_)

hatespeech_encoded_train_small = hatespeech_encoded["train"].shuffle(seed=42).select(range(train_frac))
hatespeech_encoded_test_small = hatespeech_encoded["test"].shuffle(seed=42).select(range(test_frac))

Let's print out `hatespeech_encoded_train_small` and `hatespeech_encoded_test_small` to check that everything looks alright.

---

## Transformer Models

DistilBERT is pretrained to predict masked words in a sequence of text. Let's say we have a sentence 'The house has a garden.' Then, house might be hidden with 'The ___ has a garden.' DistilBERT learns that 'house' has likely been hidden. This learning task allows the model to collect a lot of information on languages in general, but not on the particular text classification task like hate-speech detection. We need to modify the baseline language models. The next step is then to 'fine-tune' a model like DistilBERT for a particular task like in our case hate-speech analysis. Do you remember the lecture?

### Baseline

Before we investigate fine-tuning we want to quickly run a baseline model to compare our Transformer results to. In our case, this will be the dictionary-based one that we have already discussed. Run the next cell to add a column of hate_score to `df_hs`. Note that this is still a sample and not the full dataset.

Do you remember the code from the lecture? The cell puts it all together.

In [ ]:
# Keep cell
# Repeat dictionary-based sentiment analysis

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stopwords_ = list(stopwords.words('english'))
neg_words = list(set(list(pd.read_csv('https://raw.githubusercontent.com/goto4711/social-cultural-analytics/master/hate-vocab-eng.csv')['word'])))

def hate_score_(text):
    score = 0
    words = [word.lower() for word in word_tokenize(text)]
    for word in words:
        if word in neg_words:
            score += 1
    return score

df_hs = df_hs.sample(frac=frac_, random_state=1)

df_hs['hate_score'] = df_hs['tweet'].apply(hate_score_)
df_hs['hate_predict'] = np.where((df_hs['hate_score'] > 0), 'Hateful', 'Non-hateful')

df_hs["hate_predict"].value_counts(ascending=True).plot.barh()
plt.title("Frequency of Classes")
plt.show()

`hate_score_` counts the number of times a word from the hatespeech dictionary appears in the Tweets. It assigns that count to `df_hs['hate_score']`. If the hate_score is larger than 0, the Tweet is predicted to be 'Hateful' and otherwise 'Non-hateful'.

Let's calculate the accuracy:

In [ ]:
#Keep cell
from sklearn.metrics import accuracy_score

accuracy_score(df_hs["label_name"], df_hs["hate_predict"])

The accuracy of about 0.73 looks ok, but it is no better than always guessing the dominant 'Non-hateful' class, which would give about 0.73 as well. It comes almost entirely from predicting 'Non-hateful' correctly, as the confusion matrix shows. Run the cell below.

In [ ]:
#Keep cell
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(df_hs["label_name"], df_hs["hate_predict"], labels=["Non-hateful", "Hateful"])
pd.DataFrame(cm, columns = ["Non-hateful", "Hateful"], index = ["Non-hateful", "Hateful"])

> **💡 Remember** · The model pretty much fails to predict hateful Tweets and is not useful in practice. You should be very sceptical if companies just present basic accuracy to praise their models.

At least we now have a baseline of accuracy we aim to improve by fine-tuning pretrained Transformer models. Before we do this, we have to run the slightly strange looking code below, which runs the model on a GPU (or MPS for Macs) if available rather than a CPU. This makes running Transformers much faster. GPUs are the basic hardware language modelling uses to efficiently calculate billions of operations. If you are interested in the different options check out https://raoofnaushad7.medium.com/difference-between-cuda-and-mps-3d2596629c89.

Run the code below to set the default device to run our model on as either CUDA(GPU), MPS or CPU.

In [ ]:
#Keep cell

if torch.cuda.is_available():
    DEFAULT_DEVICE = "cuda"
else:
    if torch.backends.mps.is_available():
        DEFAULT_DEVICE = "mps"
    else:
        DEFAULT_DEVICE = "cpu"
device = torch.device(DEFAULT_DEVICE)
print(DEFAULT_DEVICE)

> **⚠️ Important** · It should print out the device you are using. In my case it would be 'mps' for my home computer and 'cuda' on Google Colab if you connected the runtime correctly. If you see 'cpu', start from the beginning, connect the right runtime and run all cells above again.

Let's go on and finally fine-tune.

### Fine-Tuning Transformers

We want to fine-tune the pretrained DistilBERT model and first need to get it from `AutoModelForSequenceClassification`. The Hugging Face `AutoModelForSequenceClassification` model has a so-called classification _head_ on top of the pretrained model body, which can be trained. We just need to specify how many labels the model has to predict since this dictates the number of outputs the classification head has. We set `num_labels=2` for the binary choice of hate_speech or not.

Run the cell below. The most important line is `model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2).to(device)`, which uses the `checkpoint = "distilbert-base-cased"` from the tokenizer above to load `AutoModelForSequenceClassification` for a binary classification (`num_labels=2`). `to(device)` sends the calculations to our default device, which should be cuda on Colab.

In [ ]:
#Keep cell
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2).to(device)

You might see some warnings in the following code about Hugging Face versions. Please ignore these for this session.  

The next step is to define the metrics that we'll use to evaluate our model's performance during fine-tuning. To this end, Hugging Face requires a `compute_metrics()` function for its `Trainer` that we will define later.  This function receives a `pred`, which is a  tuple with `predictions` and `label_ids` attributes. It has to return a dictionary that maps each metric's name to its value.

For our classification application, we will compute the accuracy of the model, as this allows us to compare with our baseline. Try and understand the cell below and run it. You can reuse this function or a similar one for many Hugging Face classifications. Just keep it in mind.

Run the cell below.

In [ ]:
#Keep cell
from sklearn.metrics import accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

### Training the model

To run the Hugging Face `Trainer` (https://huggingface.co/docs/transformers/main_classes/trainer), we first define some of its parameters with `TrainingArguments`. It stores a lot of information and gives you fine-grained control over training and evaluation. You can ignore most of the arguments for now, as they are used for reporting only. Just copy and paste `TrainingArguments` next time you want to use it.

The most important argument  is `output_dir`, which is where all the files from training are stored. Three more parameters are important. `batch_size` defines the number of examples to work through before updating internal model parameters. It has a lot to do with what your machine can process in its memory at any moment in time. At the beginning, it is probably a good idea not to go beyond 128 and if you have problems use 64 or 32 as batch size. Your computer will complain a lot if you get the batch size wrong. Thus, if you see strange memory errors, a good idea is always to first reduce the batch size. This generally leads to longer processing times though, because fewer records are processed at the same pass. `epochs` are the number of passes through the network. The larger the number of epochs the higher the chances for the network to learn, but it also takes much longer. For demonstration purposes, we set `epochs = 2`. Finally, `learning_rate=2e-5` sets the speed of updating parameters in deep-learning models.

Run the cell below.

In [ ]:
#Keep cell
from transformers import TrainingArguments

batch_size = 128
epochs = 2

training_args = TrainingArguments(output_dir=f"{checkpoint}-finetuned-emotion",
                                  num_train_epochs=epochs,
                                  learning_rate=2e-5,
                                  per_device_train_batch_size=batch_size,
                                  per_device_eval_batch_size=batch_size,
                                  weight_decay=0.01,
                                  eval_strategy="epoch",
                                  disable_tqdm=False,
                                  report_to="none")


We can now fine-tune our model with the `Trainer`. Check the documentation for the explanation of the arguments.

Run the next cell. This might take a while as you are now doing really advanced stuff.

In [ ]:
#Keep cell
from transformers import Trainer

trainer = Trainer(model=model,
                  args=training_args,
                  compute_metrics=compute_metrics,
                  train_dataset=hatespeech_encoded_train_small,
                  eval_dataset=hatespeech_encoded_test_small,
                  processing_class=tokenizer)
                  #tokenizer=tokenizer)
trainer.train();

Looking at the logs, we can see that our model has an accuracy on the test dataset of > 88%, which is a great improvement on our baseline dictionary-based method.

To visualise the confusion matrix next, we first need to get the predictions on the test set (`hatespeech_encoded_test_small`). The `predict()` method of the `Trainer` class returns several useful objects for evaluation. Run the cell below.

In [ ]:
#Keep cell
preds_output = trainer.predict(hatespeech_encoded_test_small)

Run another test to check that you are on the right track.

In [ ]:
# Run this cell to test your code...
assert len(preds_output) == 3, f'Test failed'
print('All tests passed!')

The output of the `predict()` contains arrays of `predictions` and `label_ids`, along with the metrics we passed to the trainer. For example, the metrics can be accessed with `preds_output.metrics`. Try it.

`preds_output` also contains the raw predictions for each class. A single prediction has one value for each class: 'Non-hateful' and 'Hateful'. Take a look at the first prediction by running the cell below.

In [ ]:
#Keep cell
preds_output.predictions[0]

You will remember the outputs of classifications we have seen before? As always we get two outputs, one for each class, and we simply choose the larger one.

We can decode the predictions and select always the most likely class by using `np.argmax()`. `np.argmax` returns the indices of the maximum values. So in my output, it would return 1 as the 1 index is larger than the 0 index. In your case, this might look different.

In my case, 1 rightly predicts a 'hateful' Tweet. We can run `np.argmax()` against all the predictions at the same time with `y_preds = np.argmax(preds_output.predictions, axis=1)`. `axis=1` indicates a row-wide comparison.



Print out the first ten y_preds next.

In [ ]:
q8.check()
#q8.hint()
#q8.solution()

Run another test.

In [ ]:
# Run this cell to test your code...
assert len(y_preds) > 810, f'Test failed'
print('All tests passed! Part 1 complete!')

With the predictions, we can plot the confusion matrix. Complete the cell below by looking back at the earlier example or using the documentation for `confusion_matrix`.

In [ ]:
#Complete cell
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_test = np.array(hatespeech_encoded_test_small["labels"])

cm = ___(y_test, y_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=___)
disp.plot()
plt.show()

In [ ]:
q9.hint()
#q9.solution()

This is much better. Now, the model is actually able to predict 'hateful' Tweets (label=1).

We can also run our own Tweet examples by using Hugging Face's pipeline. This is also called inference for language models.

Run the cell below to define `pipe` and set up a `TextClassificationPipeline`. A Hugging Face `pipeline` (https://huggingface.co/docs/transformers/en/main_classes/pipelines) allows us to bring together several steps. In this case, we 'pipe' the tokenizer into the model. Please, observe that we have to move the model back to the 'cpu' first. Otherwise, the tokenizer and model are on different devices. Can you identify in the code how we do this? It's easy.

In [ ]:
#Keep cell
from transformers import TextClassificationPipeline
pipe = TextClassificationPipeline(model=model.to('cpu'), tokenizer=tokenizer)

Let's test the pipeline with two example Tweets, taken from my test data. Run the cell below.

In [ ]:
#Keep cell
nice_tweet = "'i am looking forward to seeing it!! @user @user'"
print('Nice Tweet: ', pipe(nice_tweet, top_k=None))

bad_tweet = "black genius v white supremacy are you missing the action? the #freemasons being taught a lesson about .'."
print('Bad Tweet: ', pipe(bad_tweet, top_k=None))

As you can see, the model correctly predicts non-hateful content for `nice_tweet` with a higher score for 'LABEL_0'. It is also right for the `bad_tweet`. Try some other Tweet texts!

> **💡 Remember** · Congratulations, you now know how to train a Transformer model to classify Tweets. This can only be the beginning but hopefully you found it interesting. The most important lessons to remember are the processing steps from the Hugging Face figure above, which are very standard and repeat themselves. The steps are also repeated in the first chapter of the Transformers book with some added extra explanations: _Lewis Tunstall, Leandro von Werra, Thomas Wolf (2022): Natural Language Processing with Transformers, O'Reilly Media, Inc._ While inspired by the book, we have changed the datasets, adjusted the task and removed some additional content. But the book is a great introduction to NLP with Transformers written by the Hugging Face team. It is, however, quickly getting outdated by changes to the Hugging Face libraries.

Does this look scary? Maybe, but it is actually always the same steps. Why not ask Gemini whether it can do it? This time, do not use a code cell directly but click on the Gemini symbol. A chat interface opens. Ask Gemini to "classify from ds the label from tweet using Hugging Face's DistilBERT". It should print out the same steps we have just gone through and create the code for you. I just had to make a few simple adjustments.

---

## A Second Case Study: Do Two Sentences Mean the Same Thing --- One Recipe for Every Task

Here is a twist that shows the real power of language models. Until now, different NLP tasks needed different kinds of model:

* classification tasks (like sentiment analysis) used models that output a class label — a number;
* translation models output target-language sentences;

A different architecture was needed for each job. The famous *T5 paper* ([Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://arxiv.org/pdf/1910.10683)) showed you can collapse all of these into one idea: *text in → text out*. The trick is to turn the labels into text too. Then a single model, with a single training procedure and a single piece of code, can handle classification, translation, summarisation — everything.

Why would we want this? Because instead of maintaining many specialised models, we get one general recipe. The task simply becomes "phrase your problem as text-to-text."

We will show it with a classification task — deciding whether two sentences are paraphrases — but instead of a model that outputs `0` or `1`, we'll fine-tune T5 to literally write the word `equivalent` or `non_equivalent`. This is one of the exact tasks the T5 paper used. So, here you also learn something about reproducing papers.

The dataset is *MRPC* (the Microsoft Research Paraphrase Corpus) of pairs of sentences labelled as equivalent in meaning, or not: https://www.kaggle.com/datasets/doctri/microsoft-research-paraphrase-corpus.

> **📖 Read more** · Side note: I also chose this dataset because I used it to understand benchmarking in the new economy of enrichment of GenAI: https://journals.sagepub.com/doi/10.1177/20539517261451463. The actual experiments with MRPC did not make it into the paper because of the word-count limit but they taught me a lot about how AI benchmarking really works.

To keep the focus on the *transfer*, we repeat the same steps as before (model, tokenizer, `TrainingArguments`, `Trainer`) and reuse the `device`; only the checkpoint changes to T5 (`t5_checkpoint`, `t5_tok`, `t5_args`).

Run the next cell to set up the data.

In [ ]:
#Keep cell
from datasets import load_dataset

# "nyu-mll/glue" is the current home of the classic GLUE datasets.
mrpc = load_dataset("nyu-mll/glue", "mrpc")
mrpc

`mrpc` is a `DatasetDict` with three splits. The `test` split keeps its labels hidden (it was used for a public leaderboard), so we will train on `train` and use the `validation` split as our held-out test set. Let's look at one example by running the next cell.

In [ ]:
#Keep cell
mrpc["train"][0]

So each row gives us `sentence1`, `sentence2`, and a `label`. Try reading a few by changing the index number. You can decide for yourself whether they say the same thing, which is exactly what we will teach the model to do.

Now we turn both the input and the label into text. This is the heart of the T5 idea. We build an input string from the two sentences, and we turn the numeric label into a word: `equivalent` for `1`, `non_equivalent` for `0`. The model will be trained to read the input text and generate that label word.

Complete the next cell.

In [ ]:
#Complete cell
def make_text(batch):
    return {
        "input_text": ["mrpc sentence1: " + ___ + " sentence2: " + ___
                       for s1, s2 in zip(batch["sentence1"], batch["sentence2"])],
        "target_text": ["equivalent" if y == 1 else ___
                        for y in batch["label"]],
    }

mrpc = mrpc.___(make_text, batched=True)

print("INPUT :", mrpc["train"][0]["input_text"])
print("LABEL :", mrpc["train"][0]["target_text"], ".  This label is now text, not a number")

In [ ]:
q10.check()
#q10.hint()
#q10.solution()

It looks complicated but it is actually not very complicated. And Gemini could easily help. Formulate a prompt in your mind to turn both the input and the label into text. You can also try it if you want ...

The second part of this exercise is to show that we always repeat the same steps if we fine-tune a model with Hugging Face.

**Step 1 — Model**

In the hate-speech exercise we used `AutoModelForSequenceClassification` (a classification that outputs class numbers). Here, we use `AutoModelForSeq2SeqLM` — a text-to-text model with T5. That one change is the whole point. The model now reads text and generates text.

Complete the next cell.

In [ ]:
#Complete cell
from transformers import AutoTokenizer, ___

t5_checkpoint = "google-t5/t5-small"
t5_tok = AutoTokenizer.from_pretrained(t5_checkpoint)
model_t5 = ___.from_pretrained(t5_checkpoint).to(device)

In [ ]:
q11.check()
#q11.hint()
#q11.solution()

**Step 2 — Tokenize**

Same step as always, but now we tokenize both the input text and the target text (the label word). `text_target=` tells the tokenizer it is encoding the thing the model should learn to write.

Run the cell.

In [ ]:
#Keep cell
def encode(batch):
    enc = t5_tok(batch["input_text"], truncation=True, max_length=128)
    enc["labels"] = t5_tok(text_target=batch["target_text"], truncation=True, max_length=8)["input_ids"]
    return enc

mrpc_enc = mrpc.map(encode, batched=True, remove_columns=mrpc["train"].column_names)
mrpc_enc["train"][0].keys()

A quick check that nothing went wrong.

In [ ]:
# Run this cell to test your code...
assert mrpc_enc["train"].num_rows == 3668, 'Test failed'
print('All tests passed!')

**Step 3 — Train**

Exactly the familiar `Trainer` workflow. The only new ingredient is `DataCollatorForSeq2Seq`, which knows how to pad text labels. T5 likes a slightly higher learning rate than the classifier did, so we set one here.

Run the next cell.

In [ ]:
#Keep cell
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

t5_args = TrainingArguments(
    output_dir="t5-mrpc",
    num_train_epochs=3,
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    weight_decay=0.01,
    report_to="none",
)

collator = DataCollatorForSeq2Seq(t5_tok, model=model_t5)

trainer_t5 = Trainer(
    model_t5,
    t5_args,
    train_dataset=mrpc_enc["train"],
    data_collator=collator,
    processing_class=t5_tok,
)
trainer_t5.train()

**Step 4 — Evaluate**

Now we evaluate with the held-out validation set.

Run the next cell.

In [ ]:
#Keep cell
import numpy as np

val = mrpc["validation"]
gold = ["equivalent" if y == 1 else "non_equivalent" for y in val["label"]]

preds = []
for i in range(0, len(val), 32):
    enc = t5_tok(val["input_text"][i:i+32], return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    out = model_t5.generate(**enc, max_new_tokens=5)
    preds.extend(t5_tok.batch_decode(out, skip_special_tokens=True))

acc = np.mean([p.strip() == g for p, g in zip(preds, gold)])
print(f"Held-out accuracy: {acc:.3f}")
print("Raw text the model generated (first 5):", [p.strip() for p in preds[:5]])

The model is producing the words `equivalent` or `non_equivalent`, not class numbers. We score it by comparing those words to the true label words.

Try it on your own sentences. You can observe the model returns a word, not a number. That is the text-to-text idea made concrete.

Run the next cell.

In [ ]:
#Keep cell
def classify_pair(s1, s2):
    inp = "mrpc sentence1: " + s1 + " sentence2: " + s2
    ids = t5_tok(inp, return_tensors="pt", truncation=True, max_length=128).input_ids.to(device)
    out = model_t5.generate(ids, max_new_tokens=5)
    return t5_tok.decode(out[0], skip_special_tokens=True)

pairs = [
    ("The committee approved the new budget.", "The new budget was approved by the committee."),
    ("She is reading a book in the garden.",   "The train leaves Berlin at noon."),
]
for a, b in pairs:
    print(f"Model says: {classify_pair(a, b)!r}")
    print(f"   A: {a}")
    print(f"   B: {b}\n")

What would you change to make this SAME model and code do English -> German translation instead?

Work through the next cell to reveal the answer.

In [ ]:
q12.hint()
#q12.solution()

> **💡 Remember** · We just solved a classification problem with a text generation model, by writing the labels as text. That unification, set out in the T5 paper, is a large part of why today's general-purpose language models are so powerful.

Otherwise, this is your second full fine-tuning run: the same recipe, a brand-new dataset, and this time a result you can sanity-check yourself.

Next, let's take a look at more Transformers applications.

---

## A Tour of Transformer Applications

What we have done up to now is quite advanced. Let's take a step back and look at the Transformer models without further training/fine-tuning. We can use Hugging Face models directly to do many of the things we have seen in the lecture in a process called zero-shot learning (https://en.wikipedia.org/wiki/Zero-shot_learning).

Let's use as an example another state of the union. The text below `text_sotu_europe` is the beginning of the '2025 State of the Union Address by President von der Leyen' on the state of the European Union.

There are many things we can do with this text with Transformers directly. We don't need fine-tuning or any more 'shots' to train. Let's start with a task that we are already familiar with.

### Text Classification

For text classification/sentiment analysis, we can simply run the cell below.

> **💡 Remember** · In this part, `pipeline()` gets a few extra arguments: `model=` and `revision=` name exactly the model version stored in our shared course cache, and `device=pipe_device` makes sure the model runs on the GPU. They are only there to make the notebook faster in class. In your own work you can leave them out and simply write, for instance, `pipeline("text-classification")`. Hugging Face then picks a suitable default model for the task.

In [ ]:
#Keep cell

from transformers import pipeline

pipe_device = 0 if torch.cuda.is_available() else -1 # run the pipelines on the GPU if there is one

classifier = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", revision="714eb0f", device=pipe_device)
classifier(text_sotu_europe)

The European state of the union speech makes an ambivalent impression, as you would expect when the continent is in a state of war. The score is quite low with 0 being not positive and 1 being fully positive.

Let's investigate further and look at what kind of labels the model would attach. We can do this by adding `candidate_labels`.

Complete the below cell by creating a list of labels (Strings) that you want to test for.

In [ ]:
#Complete cell
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", revision="d7645e1", device=pipe_device)

# Choose your own categories to sort the speech into (at least two):
candidate_labels = [___]

classifier(text_sotu_europe, candidate_labels=candidate_labels)

In [ ]:
q13.check()
#q13.hint()
#q13.solution()

Very mixed results, depending on the labels you used. I tried `candidate_labels=["politics", "economy", "war", "climate", "technology"]`. Then, politics and war stand out. We only know definitely that technology does not play a big role in this opening. How do we know? It has a low probability score close to 0, while the other probabilities are all relatively close to each other.

It's time to try it yourself. Change the next cell so that you use your own input text and see what the model thinks about its sentiment.

In [ ]:
#Keep cell
#Add your own text

your_text = "Your text goes here"

from transformers import pipeline

classifier = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", revision="714eb0f", device=pipe_device)
classifier(your_text)


### Named Entity Recognition

Named Entity Recognition is also easy with Transformers. Run the next cell.

In [ ]:
#Keep cell

ner_tagger = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", revision="4c53496", aggregation_strategy="simple", device=pipe_device)
outputs = ner_tagger(text_sotu_europe)
pd.DataFrame(outputs)

Maybe somebody wants to reply to the speech. For that we move away from simple `pipeline` operations to heavier GenAI models.

### Text Generation

Let's generate the reply text and move to GenAI. Run the cell below and observe that we define a specific model for the pipeline. It is the Chat version of a very small Qwen model, open sourced by Alibaba.

> **💡 Hint** · We are using a very small version of the model so that the results might be not very good. If you want and have some time to load a much bigger model, try to upgrade the model and change the first line to `model_name = 'Qwen/Qwen2.5-1.5B-Instruct'`.


In [ ]:
#Keep cell

model_name = 'Qwen/Qwen1.5-0.5B-Chat'

generator = pipeline("text-generation", model = model_name, device=pipe_device)
response = "Dear speaker, it is great to hear your vision for Europe."
prompt = text_sotu_europe + "\n\nResponse:\n" + response
outputs = generator(prompt, max_length=400)
print(outputs[0]['generated_text'])

It's a short and not great reply, because we have not got much input text. But hopefully you can see the point.

From our case studies, you have seen that many tasks — even ones with several input fields — can be reshaped into a single piece of text for the model. Let's see next how we can implement the classification of the European SOTU speech with Qwen chat. Everything has become a matter of the right kind of prompting. Take a look at the useful explanations at https://huggingface.co/docs/transformers/en/tasks/prompting.

The next cell looks complicated but this is only because it simulates speaking to the chatbot - just like you can tell ChatGPT to tell you what the sentiment of the speech was. To this end, it uses a lot of the standard format of chatbots. Do you recognise which? And,  it uses powerful f-string and print methods in Python.

> **📖 Read more** · See whether you can understand it with the help of Gemini but definitely check out https://www.w3schools.com/python/python_string_formatting.asp. https://huggingface.co/blog/qwen-3-chat-template-deep-dive explains the Qwen chat template and why you need all those different parts.


Run the next cell.

In [ ]:
#Keep cell

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Format the prompt using the chat template
messages = [
    {"role": "system", "content": "You are a helpful assistant. Classify the sentiment of the provided text. Respond with only one word: 'positive', 'negative', or 'neutral'."},
    {"role": "user", "content": f"Text: \"{text_sotu_europe}\"\nSentiment:"}
]

# The tokenizer will apply the template and add the generation prompt for the assistant's turn
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(f"--- Chat Prompt Sent to Model ---\n{chat_prompt}\n------------------------------")

outputs = generator(
    chat_prompt,
    max_new_tokens=5,  # Just enough for "negative" or "neutral"
    pad_token_id=tokenizer.eos_token_id, # Important for some models to stop properly
    do_sample=False,
)

# Gemini explains:
# The output includes the prompt, so we need to extract only the generated part.
# The 'outputs' is a list (usually of one item unless num_return_sequences > 1).
# Your loop had a typo: `seq` should be `out`.
# for out in outputs:
#     print(f"Result: {out['generated_text']}")

generated_full_text = outputs[0]['generated_text']

# Extract only the newly generated part (the assistant's response)
# The chat_prompt already has the <|im_start|>assistant part if add_generation_prompt=True
# So, the generated text will be chat_prompt + assistant_response
# A simple way is to remove the original chat_prompt from the beginning of the generated_full_text
generated_response = generated_full_text[len(chat_prompt):].strip()

print(f"Full generated text sample: {generated_full_text[:500]}...") # Print a snippet
print(f"Extracted response: '{generated_response}'")

# Simple parsing
cleaned_response = "unknown"
cleaned_response = generated_response.lower().strip().replace(".", "")
#if cleaned_response in ["positive", "negative", "neutral"]:
#    final_sentiment = cleaned_response
#else:
    # Sometimes models add extra words, try to find the keyword
#    if "positive" in cleaned_response:
#        final_sentiment = "positive"
#    elif "negative" in cleaned_response:
#        final_sentiment = "negative"
#    elif "neutral" in cleaned_response:
#        final_sentiment = "neutral"

print(f"Parsed Sentiment: {cleaned_response}")

We can also **translate** the text into German, the native language of the president. Run the next cell.

In [ ]:
#Keep cell
messages = [
    {"role": "user",
     "content": f"Translate the following English text into German.\n\n{text_sotu_europe}"}
]
output = generator(messages, max_new_tokens=400)
print(output[0]["generated_text"][-1]["content"])

The translation is not great because it is a smaller model. But I hope you see the pattern in `messages`? Let's try something else and **summarise** the text.

Run the next cell.

In [ ]:
#Keep cell
messages = [
    {"role": "user",
     "content": f"Summarize the following speech in one sentence.\n\n{text_sotu_europe}"}
]
output = generator(messages, max_new_tokens=80)
print(output[0]["generated_text"][-1]["content"])

Please run the cell below to **answer a question** about the input text. This is another typical task called question-answering.

In [ ]:
#Keep cell
messages = [
    {"role": "user",
     "content": f"Using only this text, answer the question.\n\nText: {text_sotu_europe}\n\nQuestion: What does the Union reflect?"}
]
output = generator(messages, max_new_tokens=50)
print("Answer:", output[0]["generated_text"][-1]["content"])

Now, let's see whether you can do it yourself. Write a prompt in the right part of the message that makes the model do a task of your choice on the speech. E.g., list the main topics, write a one-line headline, or extract every country mentioned.


Complete the cell below.

In [ ]:
#Complete cell
messages = [
    {"role": "user",
     "content": f"___\n\n{text_sotu_europe}"}
]
output = generator(messages, max_new_tokens=80)
print(output[0]["generated_text"][-1]["content"])

In [ ]:
q14.check()
#q14.hint()
#q14.solution()

This was our introduction to zero-shot operations, as they do not provide any additional training data.

The next step is then to give the chatbots a few examples to learn from - in this case what is a positive, negative or neutral speech. This is also called few-shot training or 'in-context learning': https://news.mit.edu/2023/large-language-models-in-context-learning-0207.

Run the next cell.

In [ ]:
# Keep cell

# --- Few-Shot Examples ---
# It's good to have a mix of sentiments
example_1_text = "I absolutely love this speech! It's fantastic and works perfectly."
example_1_sentiment = "positive"

example_2_text = "The weather today is just okay, nothing special to report."
example_2_sentiment = "neutral"

example_3_text = "I'm very disappointed with the politician's vision. It was a terrible speech."
example_3_sentiment = "negative"

# Gemini explains: Construct the Few-Shot Prompt using Chat Template ---
# The system prompt sets the overall instruction.
# The user prompt provides the examples and the text to classify.
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Your task is to classify the sentiment of the provided text. Please respond with only one word: 'positive', 'negative', or 'neutral'."
    },
    {
        "role": "user",
        "content": f"""
Here are some examples of how to classify sentiment:

Text: "{example_1_text}"
Sentiment: {example_1_sentiment}

Text: "{example_2_text}"
Sentiment: {example_2_sentiment}

Text: "{example_3_text}"
Sentiment: {example_3_sentiment}

Now, classify the following text:
Text: "{text_sotu_europe}"
Sentiment:"""
    }
]

chat_prompt_string = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("--- Full Chat Prompt Sent to Model ---")
print(chat_prompt_string)
print("--------------------------------------")

outputs = generator(
    chat_prompt_string,
    max_new_tokens=10,  # Should be enough for "positive", "negative", or "neutral"
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
)

# Since return_full_text=False, outputs[0]['generated_text'] is only the new text

generated_response = outputs[0]['generated_text'].strip()
print(f"\n-----------------------\nRaw generated response: '{generated_response}' \n--------------------")

# Simple parsing to get one of the target words
cleaned_response = "unknown"
cleaned_response = generated_response.lower().replace(".", "").replace("!", "").strip()
print(f"\nPredicted Sentiment: {cleaned_response}")


The Qwen example showcasing text generation and prompted classification highlights the growing capabilities of instruction-tuned models, which are becoming increasingly common. Remember the *DeepLearning.AI* Python with AI introduction? While the details can be a bit complex at first, it's enough for now to know that these models exist and that there's a lot of excitement around them. Few-shot learning in particular is easy to implement and can lead to excellent results.

That's it! Your lab exercises will involve running some of the recent examples, creating a few of your own, and producing at least one fine-tuned Transformer for classification. Good luck!

> **⚠️ Important** · Run the next cell to create an automated HTML download of your notebook for submission to Canvas. Please note that this might run for a while and **make sure you have saved the notebook** before running the code. If you are asked to allow access to your Google Drive, please confirm. Make sure to check the HTML output before submitting, in particular whether the right file was transformed. Did it pick up an old file? Then rename the files and run the cell again.

In [ ]:
#Keep cell
import os

# Always grab the latest helper
if not(os.path.exists('make_html.py')):
  print('Downloading make_html.py\n')
  !wget -q https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/make_html.py

from make_html import make_html

make_html("Week4_Workshop.ipynb")